# 04 McNeil-Frey ES backtest

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.append(str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from src.pipeline import build_models

m = build_models()
loss = m['loss']
sigma_ewma = m['sigma_ewma']
hs_main, ewma_main, fhs_main, fhs_500 = m['hs_main'], m['ewma_main'], m['fhs_main'], m['fhs_500']
common_main, c2 = m['common_main'], m['c2']

In [ ]:
from src.backtests import es_backtest_mf

c = common_main
res_fhs = es_backtest_mf(loss.loc[c], fhs_main['VaR'].loc[c], fhs_main['ES'].loc[c], sigma_ewma.loc[c])
print(res_fhs)

In [ ]:
sigma_hs = loss.rolling(250).std().shift(1)

res = {
    'HS':   es_backtest_mf(loss.loc[c], hs_main['var'].loc[c],   hs_main['es'].loc[c],   sigma_hs.loc[c]),
    'EWMA': es_backtest_mf(loss.loc[c], ewma_main['VaR'].loc[c], ewma_main['ES'].loc[c], sigma_ewma.loc[c]),
    'FHS':  res_fhs}
es_table = pd.DataFrame(res).T
es_table

In [ ]:
from src.backtests import es_backtest_mf_boot

res_boot = {
    'HS':   es_backtest_mf_boot(loss.loc[c], hs_main['var'].loc[c],   hs_main['es'].loc[c],   sigma_hs.loc[c]),
    'EWMA': es_backtest_mf_boot(loss.loc[c], ewma_main['VaR'].loc[c], ewma_main['ES'].loc[c], sigma_ewma.loc[c]),
    'FHS':  es_backtest_mf_boot(loss.loc[c], fhs_main['VaR'].loc[c],  fhs_main['ES'].loc[c],  sigma_ewma.loc[c]),
}
es_boot_table = pd.DataFrame(res_boot).T
es_boot_table['p_ttest'] = es_table['p_value'] 
es_boot_table

## FHS with a 500-day window (2,422-day sample)

In [ ]:
mf_500 = es_backtest_mf(loss.loc[c2], fhs_500['VaR'].loc[c2], fhs_500['ES'].loc[c2], sigma_ewma.loc[c2])
boot_500 = es_backtest_mf_boot(loss.loc[c2], fhs_500['VaR'].loc[c2], fhs_500['ES'].loc[c2], sigma_ewma.loc[c2])
mf_500, boot_500